# Build 2026 SamplePoint camera master workbook

This notebook combines the `.xls` SamplePoint databases in:

`D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles`

into one workbook with:

- `AW120_Master`
- `Cam5_Master`
- `Import_Log`

Only columns through **Point100** are retained.

Camera assignments are based on the uploaded **2026 Master Survey**. Most databases belong entirely to one camera. Three databases contain records from both cameras:

- `chloe_database_4`
- `garrett_database_7`
- `janelle_database_5`

Those three are split **row-by-row using the field photo numbers** recorded in the Master Survey, rather than assigning the entire database to one camera. The notebook stops if a mixed-database row cannot be resolved, preventing silent camera misclassification.

The notebook uses Microsoft Excel through `pywin32`, which is appropriate here because the source files are legacy `.xls` workbooks.


In [10]:
from pathlib import Path
import re

# Folder containing all copied SamplePoint .xls files
XLS_DIR = Path(
    r"D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles"
)

# Output workbook
OUTPUT_XLSX = XLS_DIR / "2026_SamplePoint_Camera_Masters.xlsx"

# Data-integrity setting:
# True = stop if any row in a mixed-camera database cannot be assigned.
STRICT_MIXED_ASSIGNMENT = True

assert XLS_DIR.exists(), f"Input directory does not exist:\n{XLS_DIR}"

xls_files = sorted(
    [p for p in XLS_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".xls"],
    key=lambda p: [int(x) if x.isdigit() else x.lower()
                   for x in re.split(r"(\d+)", p.name)]
)

print(f"Found {len(xls_files)} XLS files")
for p in xls_files:
    print(" ", p.name)

print("\nOutput:")
print(OUTPUT_XLSX)


Found 46 XLS files
  AW120_DATABASE_1.XLS
  AW120_DATABASE_2.XLS
  AW120_DATABASE_3.XLS
  AW120_DATABASE_4.XLS
  AW120_DATABASE_5.XLS
  AW120_DATABASE_6.XLS
  AW120_DATABASE_7.XLS
  AW120_DATABASE_8.XLS
  AW120_DATABASE_9.XLS
  AW120_DATABASE_10.XLS
  AW120_DATABASE_11.XLS
  AW120_DATABASE_12.XLS
  AW120_DATABASE_13.XLS
  CAM5_DATABASE_1.XLS
  CAM5_DATABASE_2.XLS
  CAM5_DATABASE_3.XLS
  CAM5_DATABASE_4.XLS
  CAM5_DATABASE_5.XLS
  CAM5_DATABASE_6.XLS
  CAM5_DATABASE_7.XLS
  CAM5_DATABASE_8.XLS
  CAM5_DATABASE_9.XLS
  CAM5_DATABASE_10.XLS
  CAM5_DATABASE_11.XLS
  CAM5_DATABASE_12.XLS
  CAM5_DATABASE_13.XLS
  CAM5_DATABASE_14.XLS
  CAM5_DATABASE_15.XLS
  CHLOE_DATABASE_2.XLS
  CHLOE_DATABASE_3.XLS
  CHLOE_DATABASE_4.XLS
  CHLOE_DATABASE_5.XLS
  GARRETT_DATABASE_1.XLS
  GARRETT_DATABASE_2.XLS
  GARRETT_DATABASE_3.XLS
  GARRETT_DATABASE_4.XLS
  GARRETT_DATABASE_5.XLS
  GARRETT_DATABASE_6.XLS
  GARRETT_DATABASE_7.XLS
  GARRETT_DATABASE_8.XLS
  JANELLE_DATABASE_1.XLS
  JANELLE_DATABASE_2.XLS


## Camera mapping from the 2026 Master Survey

The standard `AW120_database_*` and `Cam5_database_*` files are assigned by their database prefix. The observer-named databases are mapped below.

For the three mixed-camera databases, the photo-number ranges below were taken from the uploaded Master Survey and are used to resolve each SamplePoint image row.


In [11]:
# Observer-named databases that belong entirely to one camera.
# Keys are normalized to lowercase because filenames are treated case-insensitively.

OBSERVER_DB_CAMERA = {
    # AW120
    "chloe_database_2": "AW120",
    "chloe_database_5": "AW120",
    "garrett_database_1": "AW120",
    "garrett_database_2": "AW120",
    "garrett_database_3": "AW120",
    "garrett_database_4": "AW120",
    "garrett_database_5": "AW120",
    "garrett_database_8": "AW120",
    "janelle_database_1": "AW120",
    "janelle_database_2": "AW120",
    "janelle_database_3": "AW120",
    "janelle_database_4": "AW120",

    # Camera 5
    "chloe_database_3": "Cam5",
    "garrett_database_6": "Cam5",
    "janelle_database_6": "Cam5",
}

# Mixed-camera databases.
# These photo IDs are derived from the Center/North/East/South/West photo
# fields in the uploaded 2026 Master Survey.
MIXED_PHOTO_CAMERA = {
    "chloe_database_4": {
        "AW120": set(range(9174, 9194)) | set(range(9220, 9255)),
        "Cam5":  set(range(8227, 8267)),
    },
    "garrett_database_7": {
        "AW120": set(range(9300, 9340)),
        "Cam5":  set(range(8373, 8418)),
    },
    "janelle_database_5": {
        "AW120": set(range(9255, 9295)),
        "Cam5":  set(range(8303, 8353)),
    },
}

def database_camera(db_name):
    """Return AW120, Cam5, MIXED, or None for a normalized database name."""
    db = db_name.lower()

    if db.startswith("aw120_database_"):
        return "AW120"

    if db.startswith("cam5_database_"):
        return "Cam5"

    if db in MIXED_PHOTO_CAMERA:
        return "MIXED"

    return OBSERVER_DB_CAMERA.get(db)

for db in sorted(OBSERVER_DB_CAMERA):
    print(f"{db:22s} -> {OBSERVER_DB_CAMERA[db]}")

print("\nMixed databases:")
for db, groups in MIXED_PHOTO_CAMERA.items():
    print(
        f"{db:22s} -> "
        f"AW120={len(groups['AW120'])} photos, "
        f"Cam5={len(groups['Cam5'])} photos"
    )


chloe_database_2       -> AW120
chloe_database_3       -> Cam5
chloe_database_5       -> AW120
garrett_database_1     -> AW120
garrett_database_2     -> AW120
garrett_database_3     -> AW120
garrett_database_4     -> AW120
garrett_database_5     -> AW120
garrett_database_6     -> Cam5
garrett_database_8     -> AW120
janelle_database_1     -> AW120
janelle_database_2     -> AW120
janelle_database_3     -> AW120
janelle_database_4     -> AW120
janelle_database_6     -> Cam5

Mixed databases:
chloe_database_4       -> AW120=55 photos, Cam5=40 photos
garrett_database_7     -> AW120=40 photos, Cam5=45 photos
janelle_database_5     -> AW120=40 photos, Cam5=50 photos


## Read and merge the legacy `.xls` databases

This cell:

1. opens each `.xls` with Excel;
2. locates the header containing `Point100`;
3. discards every column after `Point100`;
4. checks that all database schemas agree;
5. assigns rows to `AW120` or `Cam5`;
6. resolves the three mixed databases by matching photo numbers in the metadata portion of each row;
7. writes the two master sheets plus an import log.

No source `.xls` file is modified.


In [12]:
try:
    import win32com.client as win32
except ImportError as e:
    raise ImportError(
        "This notebook requires pywin32 because the inputs are legacy .xls files.\n"
        "Install it in this Python environment with:\n\n"
        "    pip install pywin32\n"
    ) from e


def normalize_header(value):
    """Normalize a column name for robust comparisons."""
    if value is None:
        return ""
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())


def is_blank_row(row):
    return all(v is None or str(v).strip() == "" for v in row)


def as_2d(value):
    """Normalize Excel COM UsedRange.Value into a list of lists."""
    if value is None:
        return []

    if not isinstance(value, tuple):
        return [[value]]

    if value and not isinstance(value[0], tuple):
        return [list(value)]

    return [list(r) for r in value]


def read_samplepoint_xls(excel, path):
    """
    Read the first worksheet from a SamplePoint XLS file.
    Return:
        header,
        rows,
        point1_index
    where all columns after Point100 have already been removed.
    """
    wb = None
    try:
        wb = excel.Workbooks.Open(
            Filename=str(path),
            UpdateLinks=0,
            ReadOnly=True,
            AddToMru=False,
        )

        ws = wb.Worksheets(1)
        values = as_2d(ws.UsedRange.Value)

        if not values:
            raise ValueError(f"No data found in {path.name}")

        # Search near the top of the workbook for the row containing Point100.
        header_row_index = None
        point100_index = None

        for r_i, row in enumerate(values[:25]):
            normalized = [normalize_header(v) for v in row]

            if "point100" in normalized:
                header_row_index = r_i
                point100_index = normalized.index("point100")
                break

        if header_row_index is None:
            raise ValueError(
                f"Could not locate a header containing Point100 in {path.name}"
            )

        header = list(
            values[header_row_index][:point100_index + 1]
        )

        normalized_header = [
            normalize_header(v)
            for v in header
        ]

        if "point1" not in normalized_header:
            raise ValueError(
                f"Point1 was not found in the detected header of {path.name}"
            )

        point1_index = normalized_header.index("point1")

        rows = []

        for raw_row in values[header_row_index + 1:]:

            row = list(
                raw_row[:point100_index + 1]
            )

            # Pad short rows so every row matches the header width.
            if len(row) < len(header):
                row.extend(
                    [None] * (len(header) - len(row))
                )

            if is_blank_row(row):
                continue

            # Ignore accidental repeated header rows.
            if (
                [normalize_header(v) for v in row]
                == normalized_header
            ):
                continue

            rows.append(row)

        return header, rows, point1_index

    finally:
        if wb is not None:
            wb.Close(SaveChanges=False)


def extract_matching_photo_ids(
    row,
    metadata_end_index,
    valid_photo_ids
):
    """
    Find Master-Survey photo numbers anywhere in the metadata columns
    preceding Point1.

    This works whether the XLS stores an image as a numeric ID or embeds
    the number in a filename such as DSCN9255.JPG.
    """

    hits = set()

    for value in row[:metadata_end_index]:

        if value is None:
            continue

        # Numeric cells such as 9255 or 9255.0
        if (
            isinstance(value, (int, float))
            and not isinstance(value, bool)
        ):
            try:
                number = float(value)

                if number.is_integer():

                    photo_id = int(number)

                    if photo_id in valid_photo_ids:
                        hits.add(photo_id)

            except (TypeError, ValueError):
                pass

        # Text cells / filenames containing numeric tokens.
        text = str(value)

        for token in re.findall(
            r"(?<!\d)(\d{4,6})(?!\d)",
            text
        ):
            photo_id = int(token)

            if photo_id in valid_photo_ids:
                hits.add(photo_id)

    return hits


def split_mixed_rows(
    db_name,
    rows,
    point1_index
):
    """Resolve each row in a mixed-camera database using its photo number."""

    groups = MIXED_PHOTO_CAMERA[db_name]

    photo_to_camera = {}

    for camera, ids in groups.items():

        for photo_id in ids:

            if photo_id in photo_to_camera:
                raise ValueError(
                    f"Photo ID {photo_id} occurs in multiple camera groups "
                    f"for {db_name}"
                )

            photo_to_camera[photo_id] = camera

    valid_ids = set(photo_to_camera)

    assigned = {
        "AW120": [],
        "Cam5": [],
    }

    unresolved = []

    for row_number, row in enumerate(
        rows,
        start=1
    ):

        hits = extract_matching_photo_ids(
            row=row,
            metadata_end_index=point1_index,
            valid_photo_ids=valid_ids,
        )

        cameras = {
            photo_to_camera[h]
            for h in hits
        }

        if len(cameras) == 1:

            camera = next(iter(cameras))
            assigned[camera].append(row)

        else:

            unresolved.append(
                {
                    "data_row": row_number,
                    "matched_photo_ids": sorted(hits),
                    "matched_cameras": sorted(cameras),
                    "metadata_preview": row[:point1_index],
                }
            )

    return assigned, unresolved


def write_matrix(
    ws,
    start_row,
    start_col,
    matrix
):
    """Efficient block write to an Excel worksheet."""

    if not matrix:
        return

    n_rows = len(matrix)
    n_cols = len(matrix[0])

    for row in matrix:

        if len(row) != n_cols:
            raise ValueError(
                "Attempted to write a ragged matrix to Excel."
            )

    target = ws.Range(
        ws.Cells(
            start_row,
            start_col
        ),
        ws.Cells(
            start_row + n_rows - 1,
            start_col + n_cols - 1
        ),
    )

    target.Value = tuple(
        tuple(row)
        for row in matrix
    )


# ============================================================
# INITIALIZE MASTER COLLECTIONS
# ============================================================

masters = {
    "AW120": [],
    "Cam5": [],
}

import_log = []

unresolved_mixed = []

canonical_header = None
canonical_header_norm = None
canonical_point1_index = None


# ============================================================
# OPEN EXCEL
# ============================================================

excel = win32.DispatchEx(
    "Excel.Application"
)

excel.Visible = False
excel.DisplayAlerts = False
excel.ScreenUpdating = False


try:

    # ========================================================
    # READ ALL SOURCE DATABASES
    # ========================================================

    for path in xls_files:

        db_name = path.stem.lower()

        assignment = database_camera(
            db_name
        )

        if assignment is None:

            raise ValueError(
                f"No camera mapping exists for database: {path.name}\n"
                "Add it to OBSERVER_DB_CAMERA or MIXED_PHOTO_CAMERA "
                "before continuing."
            )

        header, rows, point1_index = read_samplepoint_xls(
            excel,
            path
        )

        header_norm = [
            normalize_header(v)
            for v in header
        ]

        # ----------------------------------------------------
        # Establish and enforce one common schema.
        # ----------------------------------------------------

        if canonical_header is None:

            canonical_header = header
            canonical_header_norm = header_norm
            canonical_point1_index = point1_index

        else:

            if header_norm != canonical_header_norm:

                first_difference = next(
                    (
                        i
                        for i, (a, b)
                        in enumerate(
                            zip(
                                canonical_header_norm,
                                header_norm
                            )
                        )
                        if a != b
                    ),
                    None,
                )

                if (
                    len(header_norm)
                    != len(canonical_header_norm)
                ):

                    detail = (
                        f"column counts differ: "
                        f"canonical={len(canonical_header_norm)}, "
                        f"{path.name}={len(header_norm)}"
                    )

                elif first_difference is not None:

                    detail = (
                        f"first difference at column "
                        f"{first_difference + 1}: "
                        f"canonical="
                        f"{canonical_header[first_difference]!r}, "
                        f"{path.name}="
                        f"{header[first_difference]!r}"
                    )

                else:

                    detail = "headers differ"

                raise ValueError(
                    f"Schema mismatch in {path.name}: {detail}\n"
                    "Stopped rather than concatenating "
                    "misaligned columns."
                )

            if point1_index != canonical_point1_index:

                raise ValueError(
                    f"Point1 occurs in a different column "
                    f"in {path.name}."
                )

        # ----------------------------------------------------
        # CAMERA ASSIGNMENT
        # ----------------------------------------------------

        aw120_n = 0
        cam5_n = 0
        note = ""

        if assignment == "AW120":

            masters["AW120"].extend(rows)
            aw120_n = len(rows)

        elif assignment == "Cam5":

            masters["Cam5"].extend(rows)
            cam5_n = len(rows)

        elif assignment == "MIXED":

            assigned, unresolved = split_mixed_rows(
                db_name=db_name,
                rows=rows,
                point1_index=point1_index,
            )

            masters["AW120"].extend(
                assigned["AW120"]
            )

            masters["Cam5"].extend(
                assigned["Cam5"]
            )

            aw120_n = len(
                assigned["AW120"]
            )

            cam5_n = len(
                assigned["Cam5"]
            )

            if unresolved:

                note = (
                    f"{len(unresolved)} "
                    "unresolved mixed-camera rows"
                )

                for item in unresolved:

                    unresolved_mixed.append(
                        {
                            "database": db_name,
                            "file": path.name,
                            **item,
                        }
                    )

            else:

                note = (
                    "mixed database resolved "
                    "row-by-row"
                )

        # ----------------------------------------------------
        # IMPORT LOG
        # ----------------------------------------------------

        import_log.append(
            [
                path.name,
                db_name,
                assignment,
                len(rows),
                aw120_n,
                cam5_n,
                note,
            ]
        )

        print(
            f"{path.name:28s} "
            f"rows={len(rows):4d}  "
            f"AW120={aw120_n:4d}  "
            f"Cam5={cam5_n:4d}"
        )


    # ========================================================
    # STOP ON UNRESOLVED MIXED-CAMERA ROWS
    # ========================================================

    if unresolved_mixed:

        print(
            "\nUNRESOLVED MIXED-CAMERA ROWS"
        )

        print("-" * 80)

        for item in unresolved_mixed[:25]:

            print(
                f"{item['database']} | "
                f"data row {item['data_row']} | "
                f"photo hits="
                f"{item['matched_photo_ids']} | "
                f"camera hits="
                f"{item['matched_cameras']}"
            )

        if len(unresolved_mixed) > 25:

            print(
                f"... plus "
                f"{len(unresolved_mixed) - 25} "
                "additional unresolved rows"
            )

        if STRICT_MIXED_ASSIGNMENT:

            raise RuntimeError(
                f"\nStopped because "
                f"{len(unresolved_mixed)} row(s) "
                "in mixed-camera databases could "
                "not be assigned unambiguously.\n"
                "No master workbook was written."
            )


    # ========================================================
    # WRITE THE MASTER WORKBOOK
    # ========================================================

    if OUTPUT_XLSX.exists():
        OUTPUT_XLSX.unlink()

    out_wb = excel.Workbooks.Add()

    try:

        # ----------------------------------------------------
        # Make exactly three sheets.
        # ----------------------------------------------------

        while out_wb.Worksheets.Count < 3:
            out_wb.Worksheets.Add()

        while out_wb.Worksheets.Count > 3:
            out_wb.Worksheets(
                out_wb.Worksheets.Count
            ).Delete()

        ws_aw120 = out_wb.Worksheets(1)
        ws_cam5 = out_wb.Worksheets(2)
        ws_log = out_wb.Worksheets(3)

        ws_aw120.Name = "AW120_Master"
        ws_cam5.Name = "Cam5_Master"
        ws_log.Name = "Import_Log"


        # ====================================================
        # ADD EXPLICIT CAMERA FIELD
        # ====================================================

        master_header = [
            "Camera"
        ] + canonical_header

        aw120_output_rows = [
            ["AW120"] + row
            for row in masters["AW120"]
        ]

        cam5_output_rows = [
            ["Cam5"] + row
            for row in masters["Cam5"]
        ]


        # ----------------------------------------------------
        # Write AW120 master
        # ----------------------------------------------------

        write_matrix(
            ws_aw120,
            1,
            1,
            [master_header] + aw120_output_rows,
        )


        # ----------------------------------------------------
        # Write Camera 5 master
        # ----------------------------------------------------

        write_matrix(
            ws_cam5,
            1,
            1,
            [master_header] + cam5_output_rows,
        )


        # ====================================================
        # WRITE IMPORT LOG
        # ====================================================

        log_header = [
            "SourceFile",
            "Database",
            "DatabaseAssignment",
            "RowsRead",
            "RowsToAW120",
            "RowsToCam5",
            "Notes",
        ]

        write_matrix(
            ws_log,
            1,
            1,
            [log_header] + import_log
        )


        # ====================================================
        # BASIC FORMATTING
        # ====================================================

        for ws in (
            ws_aw120,
            ws_cam5,
            ws_log
        ):

            ws.Rows(1).Font.Bold = True
            ws.Rows(1).AutoFilter()

            ws.Application.ActiveWindow.SplitRow = 1
            ws.Application.ActiveWindow.FreezePanes = True


        # Avoid expensive full-sheet autofit
        # on the large master sheets.
        ws_log.Columns.AutoFit()


        # ====================================================
        # SAVE
        # ====================================================

        # XLSX format = 51 (xlOpenXMLWorkbook)
        out_wb.SaveAs(
            str(OUTPUT_XLSX),
            FileFormat=51
        )

    finally:

        out_wb.Close(
            SaveChanges=False
        )


finally:

    excel.ScreenUpdating = True
    excel.DisplayAlerts = True
    excel.Quit()


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n" + "=" * 72
)

print(
    "MASTER BUILD COMPLETE"
)

print(
    "=" * 72
)

print(
    f"AW120 rows: "
    f"{len(masters['AW120']):,}"
)

print(
    f"Cam5 rows:  "
    f"{len(masters['Cam5']):,}"
)

print(
    f"Columns retained through Point100: "
    f"{len(canonical_header):,}"
)

print(
    f"Output: {OUTPUT_XLSX}"
)

AW120_DATABASE_1.XLS         rows=  50  AW120=  50  Cam5=   0
AW120_DATABASE_2.XLS         rows=  50  AW120=  50  Cam5=   0
AW120_DATABASE_3.XLS         rows=  50  AW120=  50  Cam5=   0
AW120_DATABASE_4.XLS         rows=  50  AW120=  50  Cam5=   0
AW120_DATABASE_5.XLS         rows=  50  AW120=  50  Cam5=   0
AW120_DATABASE_6.XLS         rows=  50  AW120=  50  Cam5=   0
AW120_DATABASE_7.XLS         rows=  50  AW120=  50  Cam5=   0
AW120_DATABASE_8.XLS         rows=  50  AW120=  50  Cam5=   0
AW120_DATABASE_9.XLS         rows=  50  AW120=  50  Cam5=   0
AW120_DATABASE_10.XLS        rows=  50  AW120=  50  Cam5=   0
AW120_DATABASE_11.XLS        rows=  50  AW120=  50  Cam5=   0
AW120_DATABASE_12.XLS        rows=  50  AW120=  50  Cam5=   0
AW120_DATABASE_13.XLS        rows=  45  AW120=  45  Cam5=   0
CAM5_DATABASE_1.XLS          rows=  50  AW120=   0  Cam5=  50
CAM5_DATABASE_2.XLS          rows=  50  AW120=   0  Cam5=  50
CAM5_DATABASE_3.XLS          rows=  50  AW120=   0  Cam5=  50
CAM5_DAT

## Optional verification

Run this after the build cell. It opens the output workbook read-only and confirms the dimensions of the two camera sheets and the import log.


In [13]:
import win32com.client as win32

excel = win32.DispatchEx("Excel.Application")
excel.Visible = False
excel.DisplayAlerts = False

try:
    wb = excel.Workbooks.Open(
        Filename=str(OUTPUT_XLSX),
        UpdateLinks=0,
        ReadOnly=True,
        AddToMru=False,
    )

    try:
        for sheet_name in ["AW120_Master", "Cam5_Master", "Import_Log"]:
            ws = wb.Worksheets(sheet_name)
            used = ws.UsedRange

            print(
                f"{sheet_name:15s}  "
                f"rows={used.Rows.Count:,}  "
                f"columns={used.Columns.Count:,}"
            )

            # Confirm the final retained field on the two master sheets.
            if sheet_name != "Import_Log":
                last_header = ws.Cells(1, used.Columns.Count).Value
                print(f"  final column: {last_header}")

    finally:
        wb.Close(SaveChanges=False)

finally:
    excel.Quit()


AW120_Master     rows=1,266  columns=105
  final column: Point100
Cam5_Master      rows=1,016  columns=105
  final column: Point100
Import_Log       rows=47  columns=7


In [14]:
import pandas as pd

# Find the Comment column
header_norm = [normalize_header(h) for h in canonical_header]
comment_idx = header_norm.index("comment")

records = []

for camera, rows in masters.items():
    for row in rows:
        comment = row[comment_idx]

        if comment is None:
            continue

        comment = str(comment).strip()

        if not comment:
            continue

        records.append({
            "Camera": camera,
            "Comment": comment
        })

comments_df = pd.DataFrame(records)

# Summarize unique comments across both camera datasets
unique_comments = (
    comments_df
    .groupby("Comment", as_index=False)
    .agg(
        Count=("Comment", "size"),
        Cameras=("Camera", lambda x: ", ".join(sorted(set(x))))
    )
    .sort_values(["Count", "Comment"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Nonblank comment records: {len(comments_df):,}")
print(f"Unique comments: {len(unique_comments):,}")

display(unique_comments)

Nonblank comment records: 165
Unique comments: 61


,Comment,Count,Cameras
0,all other hits are DEPI,20,AW120
1,all others are HECO26,16,"AW120, Cam5"
2,all others are CHJU,11,AW120
3,all others are SAVE4,8,AW120
4,Other points; SAVE4,7,Cam5
...,...,...,...
56,"points 31, 69, 79, 80 CICY",1,AW120
57,"pt 29,57,58,89,90,97, Ambrosia acanthicarpa, p...",1,Cam5
58,"pts 39,40,49,50, Ladeania lanceolata,",1,Cam5
59,"pts 6-9, 16-18,26-28,70,ACHY",1,Cam5


In [15]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
    "display.width", None
):
    display(unique_comments)

,Comment,Count,Cameras
0,all other hits are DEPI,20,AW120
1,all others are HECO26,16,"AW120, Cam5"
2,all others are CHJU,11,AW120
3,all others are SAVE4,8,AW120
4,Other points; SAVE4,7,Cam5
5,all others are DEPI,7,"AW120, Cam5"
6,other points; CHJU,6,"AW120, Cam5"
7,other points; SAVE4,5,Cam5
8,Other points; ATCO,4,Cam5
9,all other hits are ARAR8,4,AW120


In [16]:
from pathlib import Path
import re
import shutil
import openpyxl

MASTER_XLSX = Path(
    r"D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles"
    r"\2026_SamplePoint_Camera_Masters.xlsx"
)

RGB_XLSX = MASTER_XLSX.with_name(
    "2026_SamplePoint_Camera_Masters_RGBonly.xlsx"
)

TEXT_XLSX = MASTER_XLSX.with_name(
    "2026_SamplePoint_Camera_Masters_TextOnly.xlsx"
)

# ------------------------------------------------------------
# A SamplePoint point cell is expected to look like:
#
#     OTHER, 133, 140, 122
#     DEPI, 84, 101, 72
#     ROCK, 155, 149, 136
#
# We specifically interpret the LAST three comma-separated
# integers as R, G, B.
# ------------------------------------------------------------

RGB_RE = re.compile(
    r"^(.*?)\s*,\s*"
    r"(\d{1,3})\s*,\s*"
    r"(\d{1,3})\s*,\s*"
    r"(\d{1,3})\s*$"
)


def split_samplepoint_value(value):
    """
    Split:
        'OTHER, 133, 140, 122'

    into:
        text = 'OTHER'
        rgb  = '133, 140, 122'

    Returns (None, None) if the cell cannot be interpreted.
    """

    if value is None:
        return None, None

    text = str(value).strip()

    if not text:
        return None, None

    match = RGB_RE.match(text)

    if match is None:
        return None, None

    label = match.group(1).strip()

    r = int(match.group(2))
    g = int(match.group(3))
    b = int(match.group(4))

    # RGB values must actually be valid
    if not all(0 <= x <= 255 for x in (r, g, b)):
        return None, None

    rgb = f"{r}, {g}, {b}"

    return label, rgb


# ------------------------------------------------------------
# Make exact copies first so the original master is untouched
# ------------------------------------------------------------

shutil.copy2(MASTER_XLSX, RGB_XLSX)
shutil.copy2(MASTER_XLSX, TEXT_XLSX)


# ------------------------------------------------------------
# Transform Point1 ... Point100 only
# ------------------------------------------------------------

def transform_workbook(path, mode):

    wb = openpyxl.load_workbook(path)

    transformed = 0
    blank = 0
    unresolved = []

    for sheet_name in ["AW120_Master", "Cam5_Master"]:

        ws = wb[sheet_name]

        # Find columns by header, so adding Camera does not matter.
        headers = {
            str(cell.value).strip(): cell.column
            for cell in ws[1]
            if cell.value is not None
        }

        point_cols = {
            n: headers[f"Point{n}"]
            for n in range(1, 101)
        }

        for excel_row in range(2, ws.max_row + 1):

            for point_number, col in point_cols.items():

                cell = ws.cell(excel_row, col)
                original = cell.value

                if original is None or str(original).strip() == "":
                    blank += 1
                    continue

                label, rgb = split_samplepoint_value(original)

                if label is None:
                    unresolved.append({
                        "Sheet": sheet_name,
                        "ExcelRow": excel_row,
                        "Point": point_number,
                        "Value": original,
                    })
                    continue

                if mode == "rgb":
                    cell.value = rgb

                elif mode == "text":
                    cell.value = label

                else:
                    raise ValueError(
                        "mode must be either 'rgb' or 'text'"
                    )

                transformed += 1

    wb.save(path)

    return transformed, blank, unresolved


# ------------------------------------------------------------
# RGB-only
# ------------------------------------------------------------

rgb_n, rgb_blank, rgb_unresolved = transform_workbook(
    RGB_XLSX,
    mode="rgb"
)

# ------------------------------------------------------------
# Text-only
# ------------------------------------------------------------

text_n, text_blank, text_unresolved = transform_workbook(
    TEXT_XLSX,
    mode="text"
)


print("=" * 72)
print("SAMPLEPOINT DATASETS CREATED")
print("=" * 72)

print(f"\nRGB-only Point cells transformed:  {rgb_n:,}")
print(f"Text-only Point cells transformed: {text_n:,}")

print(f"\nUnresolved RGB cells:  {len(rgb_unresolved):,}")
print(f"Unresolved Text cells: {len(text_unresolved):,}")

print("\nRGB-only:")
print(RGB_XLSX)

print("\nText-only:")
print(TEXT_XLSX)

SAMPLEPOINT DATASETS CREATED

RGB-only Point cells transformed:  183,700
Text-only Point cells transformed: 183,700

Unresolved RGB cells:  0
Unresolved Text cells: 0

RGB-only:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Camera_Masters_RGBonly.xlsx

Text-only:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Camera_Masters_TextOnly.xlsx


In [18]:
from pathlib import Path
import re
import openpyxl
import pandas as pd

INPUT_XLSX = Path(
    r"D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles"
    r"\2026_SamplePoint_Camera_Masters_TextOnly.xlsx"
)

OUTPUT_XLSX = INPUT_XLSX.with_name(
    "2026_SamplePoint_Camera_Masters_OTHER_resolved.xlsx"
)

# ------------------------------------------------------------------
# Explicit scientific-name -> USDA code mappings that we have verified
# ------------------------------------------------------------------

SCIENTIFIC_NAME_CODES = {
    "ladeania lanceolata": "PSLA3",
    "ambrosia acanthicarpa": "AMAC2",

    # Observer comment uses the older name / spelling
    "artemesia spinescens": "PIDE4",
    "artemisia spinescens": "PIDE4",
}

# Explicitly ignore these comments.
IGNORE_COMMENTS = {
    "artr2 points are dead unknown shrub",
}

# USDA PLANTS-style codes:
# examples: DEPI, CHJU, SAVE4, HECO26, ERTR13
USDA_CODE_RE = re.compile(r"\b[A-Za-z]{4}\d{0,2}\b")

# Words that can superficially look like four-letter codes but are not
# species codes in this context.
STOPWORDS = {
    "HITS",
    "DEAD",
    "ROCK",
    "SOIL",
    "BARE",
    "NONE",
}


def normalize_text(text):
    """
    Normalize spacing and a few known observer-entry typos
    without changing the original Comment field in the workbook.
    """
    text = str(text).strip()

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text)

    # Known typo: "aall other..." -> "all other..."
    text = re.sub(
        r"\baall\b",
        "all",
        text,
        flags=re.IGNORECASE
    )

    # Missing space after "are":
    # "all others areHECO26" -> "all others are HECO26"
    text = re.sub(
        r"\bare(?=[A-Za-z]{4}\d{0,2}\b)",
        "are ",
        text,
        flags=re.IGNORECASE
    )

    return text

def identify_taxon(text):
    """
    Identify a taxon from a text fragment.

    Returns a USDA code or None.

    Scientific names are checked first, then USDA-style codes.
    """
    if text is None:
        return None

    clean = normalize_text(text)
    lower = clean.lower()

    # Verified scientific names
    for scientific_name, code in SCIENTIFIC_NAME_CODES.items():
        if scientific_name in lower:
            return code

    # USDA-style codes
    candidates = []

    for match in USDA_CODE_RE.findall(clean):
        code = match.upper()

        if code not in STOPWORDS:
            candidates.append(code)

    # Only accept a single unambiguous code
    unique = sorted(set(candidates))

    if len(unique) == 1:
        return unique[0]

    return None


def expand_points(text):
    """
    Expand point specifications such as:

        31,69,79,80
        6-9, 16-18,26-28,70

    Returns sorted unique integers from 1 through 100.
    """
    points = set()

    tokens = re.findall(r"\d+\s*-\s*\d+|\d+", text)

    for token in tokens:

        if "-" in token:

            a, b = map(
                int,
                re.split(r"\s*-\s*", token)
            )

            lo, hi = sorted((a, b))

            for n in range(lo, hi + 1):
                if 1 <= n <= 100:
                    points.add(n)

        else:

            n = int(token)

            if 1 <= n <= 100:
                points.add(n)

    return sorted(points)


def parse_point_clauses(text):
    """
    Parse point-specific clauses.

    Examples
    --------
    'points 31,69,79,80 CICY'
        -> {31:CICY, 69:CICY, 79:CICY, 80:CICY}

    'pt 29,57,58,89,90,97, Ambrosia acanthicarpa, pt 91, ACHY'
        -> first six AMAC2, Point91 ACHY

    'pts 8,9,27,69, Ladeania Lanceolata, pts 12, ACHY, pts 54, PHHA'
        -> PSLA3 / ACHY / PHHA as appropriate
    """

    assignments = {}

    marker_re = re.compile(
        r"\b(?:points?|pts?|pt)\b",
        flags=re.IGNORECASE
    )

    markers = list(marker_re.finditer(text))

    for i, marker in enumerate(markers):

        start = marker.end()

        if i + 1 < len(markers):
            end = markers[i + 1].start()
        else:
            end = len(text)

        segment = text[start:end]

        # Determine where the taxon begins.
        taxon = identify_taxon(segment)

        if taxon is None:
            continue

        # We only want numbers preceding the taxon specification.
        lower_segment = segment.lower()

        taxon_positions = []

        # Scientific-name positions
        for scientific_name in SCIENTIFIC_NAME_CODES:
            pos = lower_segment.find(scientific_name)
            if pos >= 0:
                taxon_positions.append(pos)

        # USDA-code positions
        for m in USDA_CODE_RE.finditer(segment):
            if m.group(0).upper() == taxon:
                taxon_positions.append(m.start())

        if not taxon_positions:
            continue

        taxon_start = min(taxon_positions)

        point_text = segment[:taxon_start]

        for point_number in expand_points(point_text):
            assignments[point_number] = taxon

    return assignments


def parse_comment(comment):
    """
    Interpret one observer comment.

    Returns:
        blanket_code
        point_assignments
        interpretation
    """

    if comment is None:
        return None, {}, "blank"

    text = normalize_text(comment)
    lower = text.lower()

    # ----------------------------------------------------------
    # Explicit ignores
    # ----------------------------------------------------------

    if lower in IGNORE_COMMENTS:
        return None, {}, "explicitly ignored"

    # This comment describes already-classified ATCO hits as dead.
    # It does NOT tell us what OTHER means.
    if "atco hits are dead atco" in lower:
        return None, {}, "non-OTHER note"

    # ----------------------------------------------------------
    # Specific point assignments
    # ----------------------------------------------------------

    point_assignments = parse_point_clauses(text)

    # ----------------------------------------------------------
    # Blanket "all OTHER" assignment
    #
    # Includes observed typos:
    #   aall other...
    #   all othes...
    #   all other...
    #   all others...
    #   all other is...
    #   other point(s)...
    # ----------------------------------------------------------

    blanket_code = None

    blanket_patterns = [
        r"\baall\s+other(?:s|\s+hits?)?\b",
        r"\ball\s+othes\b",
        r"\ball\s+others?\b",
        r"\ball\s+other\s+hits?\b",
        r"\bother\s+points?\b",
    ]

    blanket_match = None

    for pattern in blanket_patterns:
        m = re.search(pattern, lower)

        if m:
            blanket_match = m
            break

    if blanket_match:

        # For comments with "except", only inspect the part BEFORE
        # the exception when determining the blanket species.
        before_except = re.split(
            r"\bexcept\b",
            text,
            maxsplit=1,
            flags=re.IGNORECASE
        )[0]

        blanket_code = identify_taxon(before_except)

    # ----------------------------------------------------------
    # Interpretation label
    # ----------------------------------------------------------

    if blanket_code and point_assignments:
        interpretation = "blanket + point exceptions"

    elif blanket_code:
        interpretation = "blanket"

    elif point_assignments:
        interpretation = "specific points"

    else:
        interpretation = "unresolved"

    return blanket_code, point_assignments, interpretation


# ==================================================================
# LOAD WORKBOOK
# ==================================================================

wb = openpyxl.load_workbook(INPUT_XLSX)

replacement_log = []
comment_audit = []

for sheet_name in ["AW120_Master", "Cam5_Master"]:

    ws = wb[sheet_name]

    # Header lookup
    headers = {
        str(cell.value).strip(): cell.column
        for cell in ws[1]
        if cell.value is not None
    }

    comment_col = headers["Comment"]

    point_cols = {
        n: headers[f"Point{n}"]
        for n in range(1, 101)
    }

    # --------------------------------------------------------------
    # ROW-BY-ROW
    # --------------------------------------------------------------

    for excel_row in range(2, ws.max_row + 1):

        comment = ws.cell(excel_row, comment_col).value

        if comment is None or str(comment).strip() == "":
            continue

        blanket_code, specific, interpretation = parse_comment(comment)

        n_other_before = sum(
            isinstance(ws.cell(excel_row, col).value, str)
            and ws.cell(excel_row, col).value.strip().upper() == "OTHER"
            for col in point_cols.values()
        )

        row_replacements = 0

        # ----------------------------------------------------------
        # Operate from the ORIGINAL cell values.
        #
        # Specific point assignments override the blanket assignment.
        # We ONLY alter cells whose original value is OTHER.
        # ----------------------------------------------------------

        for point_number, col in point_cols.items():

            cell = ws.cell(excel_row, col)
            original = cell.value

            if not (
                isinstance(original, str)
                and original.strip().upper() == "OTHER"
            ):
                continue

            replacement = None
            rule = None

            # Specific point assignment has priority
            if point_number in specific:

                replacement = specific[point_number]
                rule = "specific point"

            # Otherwise use blanket species
            elif blanket_code is not None:

                replacement = blanket_code
                rule = "blanket OTHER"

            if replacement is None:
                continue

            cell.value = replacement
            row_replacements += 1

            replacement_log.append({
                "Sheet": sheet_name,
                "ExcelRow": excel_row,
                "Point": point_number,
                "OldValue": original,
                "NewValue": replacement,
                "Rule": rule,
                "Comment": str(comment),
            })

        comment_audit.append({
            "Sheet": sheet_name,
            "ExcelRow": excel_row,
            "Comment": str(comment),
            "Interpretation": interpretation,
            "BlanketCode": blanket_code,
            "SpecificAssignments": (
                ", ".join(
                    f"Point{k}={v}"
                    for k, v in sorted(specific.items())
                )
                if specific else ""
            ),
            "OTHER_before": n_other_before,
            "Replacements": row_replacements,
        })


# ==================================================================
# SAVE NEW WORKBOOK
# ==================================================================

wb.save(OUTPUT_XLSX)


# ==================================================================
# AUDIT TABLES
# ==================================================================

replacement_log_df = pd.DataFrame(replacement_log)
comment_audit_df = pd.DataFrame(comment_audit)

print("=" * 80)
print("OTHER RESOLUTION COMPLETE")
print("=" * 80)

print(f"\nTotal OTHER cells replaced: {len(replacement_log_df):,}")

if len(replacement_log_df):

    print("\nReplacements by species:")
    display(
        replacement_log_df["NewValue"]
        .value_counts()
        .rename_axis("SpeciesCode")
        .reset_index(name="N")
    )

print("\nFull replacement log:")
display(replacement_log_df)

print("\nComments that still contain OTHER values but were unresolved:")

unresolved = comment_audit_df[
    (comment_audit_df["OTHER_before"] > 0) &
    (comment_audit_df["Replacements"] == 0)
].copy()

display(unresolved)

print("\nSaved corrected workbook:")
print(OUTPUT_XLSX)

OTHER RESOLUTION COMPLETE

Total OTHER cells replaced: 3,317

Replacements by species:


,SpeciesCode,N
0,SAVE4,1069
1,ERNA10,456
2,HECO26,338
3,ATCO,237
4,DISP,192
5,ARAR8,182
6,DEPI,149
7,ERTR13,130
8,CHJU,106
9,ACHY,88



Full replacement log:


,Sheet,ExcelRow,Point,OldValue,NewValue,Rule,Comment
0,AW120_Master,12,1,OTHER,SAVE4,blanket OTHER,all others are SAVE4
1,AW120_Master,12,2,OTHER,SAVE4,blanket OTHER,all others are SAVE4
2,AW120_Master,12,3,OTHER,SAVE4,blanket OTHER,all others are SAVE4
3,AW120_Master,12,4,OTHER,SAVE4,blanket OTHER,all others are SAVE4
4,AW120_Master,12,5,OTHER,SAVE4,blanket OTHER,all others are SAVE4
...,...,...,...,...,...,...,...
3312,Cam5_Master,856,23,OTHER,PSLA3,blanket OTHER,"all others are Ladeania lanceolata,"
3313,Cam5_Master,856,55,OTHER,PSLA3,blanket OTHER,"all others are Ladeania lanceolata,"
3314,Cam5_Master,856,65,OTHER,PSLA3,blanket OTHER,"all others are Ladeania lanceolata,"
3315,Cam5_Master,856,72,OTHER,PSLA3,blanket OTHER,"all others are Ladeania lanceolata,"



Comments that still contain OTHER values but were unresolved:


,Sheet,ExcelRow,Comment,Interpretation,BlanketCode,SpecificAssignments,OTHER_before,Replacements



Saved corrected workbook:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Camera_Masters_OTHER_resolved.xlsx


In [19]:
import pandas as pd
import win32com.client as win32

# ============================================================
# AUDIT DATABASE COMPLETENESS
#
# For every source XLS:
#   1. Count photo/image records
#   2. Count records with Point1 populated
#   3. Identify photos with no Point1 classification
# ============================================================

def has_value(value):
    """True for a genuinely populated Excel cell."""
    return value is not None and str(value).strip() != ""


audit_records = []
missing_records = []
orphan_point1_records = []

excel = win32.DispatchEx("Excel.Application")
excel.Visible = False
excel.DisplayAlerts = False
excel.ScreenUpdating = False

try:

    for path in xls_files:

        db_name = path.stem.lower()

        # Read using the same function used for the master build
        header, rows, point1_index = read_samplepoint_xls(
            excel,
            path
        )

        header_norm = [
            normalize_header(h)
            for h in header
        ]

        # ----------------------------------------------------
        # Locate the photo identifier column.
        #
        # SamplePoint databases generally call this "image",
        # but allow photo/photos as fallbacks.
        # ----------------------------------------------------

        photo_index = None
        photo_field = None

        for candidate in ["image", "photo", "photos"]:

            if candidate in header_norm:
                photo_index = header_norm.index(candidate)
                photo_field = header[photo_index]
                break

        if photo_index is None:
            raise ValueError(
                f"Could not find image/photo field in {path.name}"
            )

        # Optional metadata fields for easier debugging
        key_index = (
            header_norm.index("key")
            if "key" in header_norm
            else None
        )

        comment_index = (
            header_norm.index("comment")
            if "comment" in header_norm
            else None
        )

        # ----------------------------------------------------
        # Count row states
        # ----------------------------------------------------

        n_photos = 0
        n_point1 = 0
        n_complete = 0
        n_missing_point1 = 0
        n_point1_without_photo = 0

        seen_photos = []

        for data_row, row in enumerate(rows, start=1):

            photo = row[photo_index]
            point1 = row[point1_index]

            photo_present = has_value(photo)
            point1_present = has_value(point1)

            if photo_present:
                n_photos += 1
                seen_photos.append(str(photo).strip())

            if point1_present:
                n_point1 += 1

            # Expected completed record
            if photo_present and point1_present:
                n_complete += 1

            # Candidate incomplete classification
            elif photo_present and not point1_present:

                n_missing_point1 += 1

                missing_records.append({
                    "Database": path.stem,
                    "CameraAssignment": database_camera(db_name),
                    "DataRow": data_row,
                    "Key": (
                        row[key_index]
                        if key_index is not None
                        else None
                    ),
                    "Photo": photo,
                    "Comment": (
                        row[comment_index]
                        if comment_index is not None
                        else None
                    ),
                })

            # Strange converse case
            elif point1_present and not photo_present:

                n_point1_without_photo += 1

                orphan_point1_records.append({
                    "Database": path.stem,
                    "DataRow": data_row,
                    "Point1": point1,
                })

        # ----------------------------------------------------
        # Duplicate photo identifiers
        # ----------------------------------------------------

        n_unique_photos = len(set(seen_photos))

        n_duplicate_photo_rows = (
            len(seen_photos) - n_unique_photos
        )

        # ----------------------------------------------------
        # Overall database status
        # ----------------------------------------------------

        counts_match = (
            n_photos == n_point1
        )

        completion_pct = (
            (n_complete / n_photos) * 100
            if n_photos
            else float("nan")
        )

        audit_records.append({
            "Database": path.stem,
            "CameraAssignment": database_camera(db_name),
            "Photos": n_photos,
            "UniquePhotos": n_unique_photos,
            "Point1Hits": n_point1,
            "CompleteRows": n_complete,
            "MissingPoint1": n_missing_point1,
            "Point1WithoutPhoto": n_point1_without_photo,
            "DuplicatePhotoRows": n_duplicate_photo_rows,
            "CompletionPct": completion_pct,
            "CountsMatch": counts_match,
        })

finally:

    excel.ScreenUpdating = True
    excel.DisplayAlerts = True
    excel.Quit()


# ============================================================
# RESULTS
# ============================================================

audit_df = pd.DataFrame(audit_records)

# Show incomplete/problem databases first
audit_df = (
    audit_df
    .sort_values(
        ["CountsMatch", "MissingPoint1", "Database"],
        ascending=[True, False, True]
    )
    .reset_index(drop=True)
)

# Cleaner percentage display
audit_df["CompletionPct"] = (
    audit_df["CompletionPct"]
    .round(1)
)

print("=" * 80)
print("SAMPLEPOINT DATABASE COMPLETENESS AUDIT")
print("=" * 80)

print(f"\nDatabases checked: {len(audit_df)}")
print(
    f"Databases with matching photo / Point1 counts: "
    f"{audit_df['CountsMatch'].sum()}"
)
print(
    f"Databases with discrepancies: "
    f"{(~audit_df['CountsMatch']).sum()}"
)

display(audit_df)


# ============================================================
# SHOW THE ACTUAL INCOMPLETE PHOTOS
# ============================================================

missing_point1_df = pd.DataFrame(missing_records)

print("\n" + "=" * 80)
print("PHOTOS WITH NO POINT1 CLASSIFICATION")
print("=" * 80)

print(
    f"\nTotal candidate incomplete photos: "
    f"{len(missing_point1_df)}"
)

if len(missing_point1_df):

    display(
        missing_point1_df
        .sort_values(
            ["Database", "DataRow"]
        )
        .reset_index(drop=True)
    )

else:

    print("None found.")


# ============================================================
# CHECK FOR THE REVERSE PROBLEM
# ============================================================

orphan_point1_df = pd.DataFrame(
    orphan_point1_records
)

if len(orphan_point1_df):

    print("\n" + "=" * 80)
    print("POINT1 VALUES WITHOUT A PHOTO IDENTIFIER")
    print("=" * 80)

    display(orphan_point1_df)

SAMPLEPOINT DATABASE COMPLETENESS AUDIT

Databases checked: 46
Databases with matching photo / Point1 counts: 33
Databases with discrepancies: 13


,Database,CameraAssignment,Photos,UniquePhotos,Point1Hits,CompleteRows,MissingPoint1,Point1WithoutPhoto,DuplicatePhotoRows,CompletionPct,CountsMatch
0,JANELLE_DATABASE_5,MIXED,90,90,10,10,80,0,0,11.1,False
1,GARRETT_DATABASE_8,AW120,80,80,20,20,60,0,0,25.0,False
2,JANELLE_DATABASE_6,Cam5,65,65,15,15,50,0,0,23.1,False
3,CAM5_DATABASE_3,Cam5,50,50,5,5,45,0,0,10.0,False
4,JANELLE_DATABASE_4,AW120,55,55,19,19,36,0,0,34.5,False
5,CAM5_DATABASE_1,Cam5,50,50,20,20,30,0,0,40.0,False
6,CAM5_DATABASE_2,Cam5,50,50,25,25,25,0,0,50.0,False
7,CAM5_DATABASE_4,Cam5,50,50,25,25,25,0,0,50.0,False
8,CAM5_DATABASE_5,Cam5,50,50,25,25,25,0,0,50.0,False
9,CAM5_DATABASE_6,Cam5,50,50,25,25,25,0,0,50.0,False



PHOTOS WITH NO POINT1 CLASSIFICATION

Total candidate incomplete photos: 443


,Database,CameraAssignment,DataRow,Key,Photo,Comment
0,CAM5_DATABASE_1,Cam5,1,1,DSCN8418_c.jpg,None
1,CAM5_DATABASE_1,Cam5,2,2,DSCN8419_c.jpg,None
2,CAM5_DATABASE_1,Cam5,3,3,DSCN8420_c.jpg,None
3,CAM5_DATABASE_1,Cam5,4,4,DSCN8421_c.jpg,None
4,CAM5_DATABASE_1,Cam5,5,5,DSCN8422_c.jpg,None
...,...,...,...,...,...,...
438,JANELLE_DATABASE_6,Cam5,46,46,DSCN8448_c.jpg,None
439,JANELLE_DATABASE_6,Cam5,47,47,DSCN8449_c.jpg,None
440,JANELLE_DATABASE_6,Cam5,48,48,DSCN8450_c.jpg,None
441,JANELLE_DATABASE_6,Cam5,49,49,DSCN8451_c.jpg,None


In [20]:
# Show every incomplete record in full

if len(missing_point1_df):

    incomplete_full = (
        missing_point1_df
        .sort_values(
            ["Database", "DataRow"]
        )
        .reset_index(drop=True)
    )

    print(
        f"Total incomplete photos: "
        f"{len(incomplete_full):,}"
    )

    print(
        f"Databases containing incompletes: "
        f"{incomplete_full['Database'].nunique():,}"
    )

    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.width", None,
    ):
        display(incomplete_full)

else:
    print("No incomplete photos found.")

Total incomplete photos: 443
Databases containing incompletes: 13


,Database,CameraAssignment,DataRow,Key,Photo,Comment
0,CAM5_DATABASE_1,Cam5,1,1,DSCN8418_c.jpg,None
1,CAM5_DATABASE_1,Cam5,2,2,DSCN8419_c.jpg,None
2,CAM5_DATABASE_1,Cam5,3,3,DSCN8420_c.jpg,None
3,CAM5_DATABASE_1,Cam5,4,4,DSCN8421_c.jpg,None
4,CAM5_DATABASE_1,Cam5,5,5,DSCN8422_c.jpg,None
5,CAM5_DATABASE_1,Cam5,6,6,DSCN8468_c.jpg,None
6,CAM5_DATABASE_1,Cam5,7,7,DSCN8469_c.jpg,None
7,CAM5_DATABASE_1,Cam5,8,8,DSCN8470_c.jpg,None
8,CAM5_DATABASE_1,Cam5,9,9,DSCN8471_c.jpg,None
9,CAM5_DATABASE_1,Cam5,10,10,DSCN8472_c.jpg,None


In [22]:
partial_databases = sorted(
    missing_point1_df["Database"]
    .dropna()
    .unique()
)

print(f"Databases with incompletes: {len(partial_databases)}")

for db in partial_databases:
    print(db)

    partial_summary = (
    missing_point1_df
    .groupby("Database")
    .size()
    .reset_index(name="IncompletePhotos")
    .sort_values("IncompletePhotos", ascending=False)
)

display(partial_summary)

Databases with incompletes: 13
CAM5_DATABASE_1
CAM5_DATABASE_2
CAM5_DATABASE_3
CAM5_DATABASE_4
CAM5_DATABASE_5
CAM5_DATABASE_6
CAM5_DATABASE_7
CHLOE_DATABASE_5
GARRETT_DATABASE_8
JANELLE_DATABASE_3
JANELLE_DATABASE_4
JANELLE_DATABASE_5
JANELLE_DATABASE_6


,Database,IncompletePhotos
11,JANELLE_DATABASE_5,80
8,GARRETT_DATABASE_8,60
12,JANELLE_DATABASE_6,50
2,CAM5_DATABASE_3,45
10,JANELLE_DATABASE_4,36
0,CAM5_DATABASE_1,30
4,CAM5_DATABASE_5,25
1,CAM5_DATABASE_2,25
3,CAM5_DATABASE_4,25
5,CAM5_DATABASE_6,25


In [23]:
from pathlib import Path
import shutil
import pandas as pd

# ============================================================
# SOURCE CAMERA DATABASE TREES
# ============================================================

CAMERA_ROOTS = {
    "AW120": Path(
        r"D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\AW120\Databases"
    ),
    "Cam5": Path(
        r"D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\Cam5\Databases"
    ),
}

# ============================================================
# OUTPUT
# ============================================================

OUTPUT_ROOT = Path(
    r"D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\Incomplete_Photos"
)

OUTPUT_DIRS = {
    camera: OUTPUT_ROOT / camera
    for camera in CAMERA_ROOTS
}

for folder in OUTPUT_DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# BUILD AN INDEX OF ALL CAMERA-DATABASE PHOTOS
#
# Filename matching is case-insensitive.
# ============================================================

photo_index = {}

for camera, root in CAMERA_ROOTS.items():

    if not root.exists():
        raise FileNotFoundError(
            f"Camera database directory does not exist:\n{root}"
        )

    print(f"Indexing {camera}:")
    print(root)

    n_files = 0

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        # Only likely image types
        if path.suffix.lower() not in {
            ".jpg", ".jpeg", ".png",
            ".tif", ".tiff"
        }:
            continue

        key = path.name.lower()

        photo_index.setdefault(key, []).append(
            {
                "Camera": camera,
                "Path": path,
            }
        )

        n_files += 1

    print(f"  indexed {n_files:,} images")


# ============================================================
# COPY INCOMPLETE PHOTOS
# ============================================================

copy_log = []

for _, row in missing_point1_df.iterrows():

    database = row["Database"]
    photo_value = row["Photo"]

    if pd.isna(photo_value):
        continue

    # SamplePoint Image field should generally contain the filename.
    # Path(...).name protects us if it contains a partial/full path.
    photo_name = Path(
        str(photo_value).strip()
    ).name

    matches = photo_index.get(
        photo_name.lower(),
        []
    )

    # --------------------------------------------------------
    # NOT FOUND
    # --------------------------------------------------------

    if len(matches) == 0:

        copy_log.append({
            "Database": database,
            "Photo": photo_name,
            "Status": "NOT FOUND",
            "Camera": None,
            "Source": None,
            "Destination": None,
        })

        continue

    # --------------------------------------------------------
    # MULTIPLE MATCHES
    #
    # Do not guess. Record them for inspection.
    # --------------------------------------------------------

    if len(matches) > 1:

        copy_log.append({
            "Database": database,
            "Photo": photo_name,
            "Status": f"MULTIPLE MATCHES ({len(matches)})",
            "Camera": ", ".join(
                sorted(set(m["Camera"] for m in matches))
            ),
            "Source": " | ".join(
                str(m["Path"])
                for m in matches
            ),
            "Destination": None,
        })

        continue

    # --------------------------------------------------------
    # UNIQUE MATCH
    # --------------------------------------------------------

    match = matches[0]

    camera = match["Camera"]
    source = match["Path"]

    destination = (
        OUTPUT_DIRS[camera]
        / source.name
    )

    # Protect against same-name collisions in output.
    if destination.exists():

        # If the same source photo has already been copied,
        # no need to create another copy.
        try:
            same_size = (
                destination.stat().st_size
                == source.stat().st_size
            )
        except OSError:
            same_size = False

        if same_size:

            status = "ALREADY COPIED"

        else:

            # Preserve both rather than overwrite.
            destination = (
                OUTPUT_DIRS[camera]
                / f"{source.stem}_{database}{source.suffix}"
            )

            shutil.copy2(
                source,
                destination
            )

            status = "COPIED — RENAMED COLLISION"

    else:

        shutil.copy2(
            source,
            destination
        )

        status = "COPIED"

    copy_log.append({
        "Database": database,
        "Photo": photo_name,
        "Status": status,
        "Camera": camera,
        "Source": str(source),
        "Destination": str(destination),
    })


# ============================================================
# AUDIT
# ============================================================

copy_log_df = pd.DataFrame(copy_log)

print("\n" + "=" * 80)
print("INCOMPLETE PHOTO EXTRACTION COMPLETE")
print("=" * 80)

print(f"\nIncomplete records requested: {len(missing_point1_df):,}")
print(f"Records processed:            {len(copy_log_df):,}")

if len(copy_log_df):

    print("\nStatus:")
    display(
        copy_log_df["Status"]
        .value_counts()
        .rename_axis("Status")
        .reset_index(name="N")
    )

    print("\nBy camera:")
    display(
        copy_log_df[
            copy_log_df["Camera"].isin(["AW120", "Cam5"])
        ]
        .groupby("Camera")
        .size()
        .reset_index(name="Photos")
    )


# ============================================================
# SHOW PROBLEMS
# ============================================================

problems = copy_log_df[
    ~copy_log_df["Status"].isin(
        ["COPIED", "ALREADY COPIED"]
    )
].copy()

if len(problems):

    print("\n" + "=" * 80)
    print("FILES REQUIRING REVIEW")
    print("=" * 80)

    with pd.option_context(
        "display.max_rows", None,
        "display.max_colwidth", None,
    ):
        display(problems)

else:

    print("\nEvery incomplete photo was uniquely located.")


# ============================================================
# SAVE COPY AUDIT
# ============================================================

audit_path = (
    OUTPUT_ROOT
    / "Incomplete_Photo_Copy_Audit.csv"
)

copy_log_df.to_csv(
    audit_path,
    index=False
)

print("\nOutput folders:")
print(OUTPUT_DIRS["AW120"])
print(OUTPUT_DIRS["Cam5"])

print("\nAudit:")
print(audit_path)

Indexing AW120:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\AW120\Databases
  indexed 645 images
Indexing Cam5:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\Cam5\Databases
  indexed 740 images

INCOMPLETE PHOTO EXTRACTION COMPLETE

Incomplete records requested: 443
Records processed:            443

Status:


,Status,N
0,COPIED,248
1,NOT FOUND,195



By camera:


,Camera,Photos
0,Cam5,248



FILES REQUIRING REVIEW


,Database,Photo,Status,Camera,Source,Destination
200,CHLOE_DATABASE_5,DSCN9435_c.jpg,NOT FOUND,NaN,NaN,NaN
201,CHLOE_DATABASE_5,DSCN9436_c.jpg,NOT FOUND,NaN,NaN,NaN
202,CHLOE_DATABASE_5,DSCN9437_c.jpg,NOT FOUND,NaN,NaN,NaN
203,CHLOE_DATABASE_5,DSCN9438_c.jpg,NOT FOUND,NaN,NaN,NaN
204,CHLOE_DATABASE_5,DSCN9439_c.jpg,NOT FOUND,NaN,NaN,NaN
205,GARRETT_DATABASE_8,DSCN9435_c.jpg,NOT FOUND,NaN,NaN,NaN
206,GARRETT_DATABASE_8,DSCN9436_c.jpg,NOT FOUND,NaN,NaN,NaN
207,GARRETT_DATABASE_8,DSCN9437_c.jpg,NOT FOUND,NaN,NaN,NaN
208,GARRETT_DATABASE_8,DSCN9438_c.jpg,NOT FOUND,NaN,NaN,NaN
209,GARRETT_DATABASE_8,DSCN9439_c.jpg,NOT FOUND,NaN,NaN,NaN



Output folders:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\Incomplete_Photos\AW120
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\Incomplete_Photos\Cam5

Audit:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\Incomplete_Photos\Incomplete_Photo_Copy_Audit.csv


In [24]:
from pathlib import Path
import shutil
import pandas as pd

# ============================================================
# EXISTING TARGET
# ============================================================

OUTPUT_ROOT = Path(
    r"D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\Incomplete_Photos"
)

OUTPUT_DIRS = {
    "AW120": OUTPUT_ROOT / "AW120",
    "Cam5": OUTPUT_ROOT / "Cam5",
}

for folder in OUTPUT_DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# SAMPLEPOINT DATABASE ARCHIVE
# ============================================================

ARCHIVE_ROOT = Path(
    r"D:\My Drive\BOP_OCTC_2025\Samplepoint\2026 Database Archive"
)

if not ARCHIVE_ROOT.exists():
    raise FileNotFoundError(
        f"Archive directory does not exist:\n{ARCHIVE_ROOT}"
    )


# ============================================================
# HELPERS
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".tif", ".tiff"
}


def normalize_db_name(value):
    if value is None:
        return None

    text = str(value).strip().lower()

    # Remove extension if present
    if text.endswith(".xls"):
        text = text[:-4]
    elif text.endswith(".xlsx"):
        text = text[:-5]

    return text


def determine_camera(database_name, photo_name):
    """
    Determine camera for a missing record.

    First use database_camera() where possible.
    Mixed databases are resolved by the photo ID using
    MIXED_PHOTO_CAMERA.
    """
    db = normalize_db_name(database_name)

    assignment = database_camera(db)

    if assignment in {"AW120", "Cam5"}:
        return assignment

    if assignment == "MIXED":

        # Extract numeric photo ID from filename
        import re

        matches = re.findall(
            r"(?<!\d)(\d{4,6})(?!\d)",
            str(photo_name)
        )

        if not matches:
            return None

        photo_id = int(matches[-1])

        groups = MIXED_PHOTO_CAMERA[db]

        for camera, ids in groups.items():
            if photo_id in ids:
                return camera

    return None


# ============================================================
# INDEX ARCHIVE DATABASE FOLDERS
# ============================================================

archive_folders = {}

for child in ARCHIVE_ROOT.iterdir():

    if not child.is_dir():
        continue

    key = normalize_db_name(child.name)

    archive_folders[key] = child


print(f"Archive database folders found: {len(archive_folders)}")


# ============================================================
# SEARCH FOR EACH INCOMPLETE PHOTO
# ============================================================

archive_copy_log = []

for _, row in missing_point1_df.iterrows():

    database = normalize_db_name(
        row["Database"]
    )

    photo_value = row["Photo"]

    if pd.isna(photo_value):
        continue

    photo_name = Path(
        str(photo_value).strip()
    ).name

    # --------------------------------------------------------
    # Find the corresponding observer database folder
    # --------------------------------------------------------

    db_folder = archive_folders.get(
        database
    )

    if db_folder is None:

        archive_copy_log.append({
            "Database": database,
            "Photo": photo_name,
            "Status": "DATABASE FOLDER NOT FOUND",
            "Camera": None,
            "Source": None,
            "Destination": None,
        })

        continue

    # --------------------------------------------------------
    # Search only inside that database folder
    # --------------------------------------------------------

    matches = [
        p
        for p in db_folder.rglob("*")
        if (
            p.is_file()
            and p.suffix.lower() in IMAGE_EXTENSIONS
            and p.name.lower() == photo_name.lower()
        )
    ]

    # --------------------------------------------------------
    # No photo found
    # --------------------------------------------------------

    if len(matches) == 0:

        archive_copy_log.append({
            "Database": database,
            "Photo": photo_name,
            "Status": "PHOTO NOT FOUND",
            "Camera": None,
            "Source": str(db_folder),
            "Destination": None,
        })

        continue

    # --------------------------------------------------------
    # Multiple matches
    # --------------------------------------------------------

    if len(matches) > 1:

        archive_copy_log.append({
            "Database": database,
            "Photo": photo_name,
            "Status": f"MULTIPLE MATCHES ({len(matches)})",
            "Camera": None,
            "Source": " | ".join(
                str(p)
                for p in matches
            ),
            "Destination": None,
        })

        continue

    # --------------------------------------------------------
    # Unique photo
    # --------------------------------------------------------

    source = matches[0]

    camera = determine_camera(
        database,
        photo_name
    )

    if camera not in {"AW120", "Cam5"}:

        archive_copy_log.append({
            "Database": database,
            "Photo": photo_name,
            "Status": "CAMERA UNRESOLVED",
            "Camera": camera,
            "Source": str(source),
            "Destination": None,
        })

        continue

    destination = (
        OUTPUT_DIRS[camera]
        / source.name
    )

    # --------------------------------------------------------
    # Avoid destructive overwrite
    # --------------------------------------------------------

    if destination.exists():

        # If same size, treat as already present
        try:
            same_size = (
                destination.stat().st_size
                == source.stat().st_size
            )
        except OSError:
            same_size = False

        if same_size:

            status = "ALREADY PRESENT"

        else:

            destination = (
                OUTPUT_DIRS[camera]
                / f"{source.stem}_{database}{source.suffix}"
            )

            shutil.copy2(
                source,
                destination
            )

            status = "COPIED — RENAMED COLLISION"

    else:

        shutil.copy2(
            source,
            destination
        )

        status = "COPIED"

    archive_copy_log.append({
        "Database": database,
        "Photo": photo_name,
        "Status": status,
        "Camera": camera,
        "Source": str(source),
        "Destination": str(destination),
    })


# ============================================================
# AUDIT
# ============================================================

archive_copy_log_df = pd.DataFrame(
    archive_copy_log
)

print("\n" + "=" * 80)
print("ARCHIVE PHOTO EXTRACTION COMPLETE")
print("=" * 80)

print(
    f"\nIncomplete records checked: "
    f"{len(archive_copy_log_df):,}"
)

if len(archive_copy_log_df):

    print("\nStatus:")
    display(
        archive_copy_log_df["Status"]
        .value_counts()
        .rename_axis("Status")
        .reset_index(name="N")
    )

    print("\nBy camera:")
    display(
        archive_copy_log_df[
            archive_copy_log_df["Camera"]
            .isin(["AW120", "Cam5"])
        ]
        .groupby("Camera")
        .size()
        .reset_index(name="Photos")
    )


# ============================================================
# SHOW ANYTHING STILL UNRESOLVED
# ============================================================

problems = archive_copy_log_df[
    ~archive_copy_log_df["Status"].isin(
        [
            "COPIED",
            "ALREADY PRESENT",
        ]
    )
].copy()

if len(problems):

    print("\n" + "=" * 80)
    print("ARCHIVE RECORDS REQUIRING REVIEW")
    print("=" * 80)

    with pd.option_context(
        "display.max_rows", None,
        "display.max_colwidth", None,
    ):
        display(problems)

else:

    print(
        "\nEvery matching archive photo was "
        "successfully added or was already present."
    )


# ============================================================
# SAVE AUDIT
# ============================================================

audit_path = (
    OUTPUT_ROOT
    / "Incomplete_Photo_Archive_Copy_Audit.csv"
)

archive_copy_log_df.to_csv(
    audit_path,
    index=False
)

print("\nAudit saved to:")
print(audit_path)

Archive database folders found: 17

ARCHIVE PHOTO EXTRACTION COMPLETE

Incomplete records checked: 443

Status:


,Status,N
0,COPIED,238
1,DATABASE FOLDER NOT FOUND,205



By camera:


,Camera,Photos
0,AW120,138
1,Cam5,100



ARCHIVE RECORDS REQUIRING REVIEW


,Database,Photo,Status,Camera,Source,Destination
0,cam5_database_1,DSCN8418_c.jpg,DATABASE FOLDER NOT FOUND,NaN,NaN,NaN
1,cam5_database_1,DSCN8419_c.jpg,DATABASE FOLDER NOT FOUND,NaN,NaN,NaN
2,cam5_database_1,DSCN8420_c.jpg,DATABASE FOLDER NOT FOUND,NaN,NaN,NaN
3,cam5_database_1,DSCN8421_c.jpg,DATABASE FOLDER NOT FOUND,NaN,NaN,NaN
4,cam5_database_1,DSCN8422_c.jpg,DATABASE FOLDER NOT FOUND,NaN,NaN,NaN
5,cam5_database_1,DSCN8468_c.jpg,DATABASE FOLDER NOT FOUND,NaN,NaN,NaN
6,cam5_database_1,DSCN8469_c.jpg,DATABASE FOLDER NOT FOUND,NaN,NaN,NaN
7,cam5_database_1,DSCN8470_c.jpg,DATABASE FOLDER NOT FOUND,NaN,NaN,NaN
8,cam5_database_1,DSCN8471_c.jpg,DATABASE FOLDER NOT FOUND,NaN,NaN,NaN
9,cam5_database_1,DSCN8472_c.jpg,DATABASE FOLDER NOT FOUND,NaN,NaN,NaN



Audit saved to:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\Incomplete_Photos\Incomplete_Photo_Archive_Copy_Audit.csv


In [29]:
from pathlib import Path
import re
import pandas as pd


# ============================================================
# MASTER SURVEY
# ============================================================

MASTER_SURVEY = Path(
    r"D:\My Drive\BOP_OCTC_2025\2026 Master Survey.xlsx"
)

SHEET_NAME = "Sheet1"


# ============================================================
# NORMALIZATION HELPERS
# ============================================================

def normalize_database(value):
    """
    Normalize database names for matching.

    Examples:
        CAM5_DATABASE_1      -> cam5_database_1
        cam5_database_1.xls  -> cam5_database_1
        JANELLE_DATABASE_4   -> janelle_database_4
    """
    if value is None or pd.isna(value):
        return None

    text = str(value).strip().lower()

    text = re.sub(
        r"\.xlsx?$",
        "",
        text
    )

    return text


def normalize_camera(value):
    """
    Normalize camera descriptions to:
        AW120
        Cam5
    """
    if value is None or pd.isna(value):
        return None

    text = str(value).strip().lower()

    if "aw120" in text:
        return "AW120"

    if (
        "cam5" in text
        or "camera 5" in text
        or "camera5" in text
    ):
        return "Cam5"

    return None


def extract_photo_number(value):
    """
    Convert:
        DSCN8418_c.jpg -> 8418
        8418           -> 8418
        8418.0         -> 8418
    """
    if value is None or pd.isna(value):
        return None

    # Numeric survey value
    if isinstance(value, (int, float)):

        try:
            n = float(value)

            if n.is_integer():
                return int(n)

        except (TypeError, ValueError):
            pass

    # Filename or other text
    match = re.search(
        r"(?<!\d)(\d{4,6})(?!\d)",
        str(value)
    )

    if match:
        return int(match.group(1))

    return None


# ============================================================
# READ SHEET1
# ============================================================

survey = pd.read_excel(
    MASTER_SURVEY,
    sheet_name=SHEET_NAME
)

PHOTO_COLUMNS = [
    "Center Photoplot",
    "North Photoplot",
    "East Photoplot",
    "South Photoplot",
    "West Photoplot",
]


# ============================================================
# RESOLVE PLOT ID
#
# Prefer Plot ID; fall back to Plot ID Incidental.
# ============================================================

def resolve_plot_id(row):

    plot_id = row.get("Plot ID")

    if (
        pd.notna(plot_id)
        and str(plot_id).strip()
    ):
        return str(plot_id).strip()

    incidental = row.get(
        "Plot ID Incidental"
    )

    if (
        pd.notna(incidental)
        and str(incidental).strip()
    ):
        return str(incidental).strip()

    return None


survey["ResolvedPlotID"] = survey.apply(
    resolve_plot_id,
    axis=1
)

survey["DatabaseNorm"] = (
    survey["Samplepoint Database Name"]
    .map(normalize_database)
)

survey["CameraNorm"] = (
    survey["Camera Number"]
    .map(normalize_camera)
)


# ============================================================
# TURN THE FIVE PHOTO FIELDS INTO ONE PHOTO LOOKUP TABLE
# ============================================================

photo_lookup = survey.melt(
    id_vars=[
        "ObjectID",
        "ResolvedPlotID",
        "Plot ID",
        "Plot ID Incidental",
        "Samplepoint Database Name",
        "DatabaseNorm",
        "Camera Number",
        "CameraNorm",
    ],
    value_vars=PHOTO_COLUMNS,
    var_name="PhotoPosition",
    value_name="SurveyPhotoNumber"
)

photo_lookup["PhotoNumber"] = (
    photo_lookup["SurveyPhotoNumber"]
    .map(extract_photo_number)
)

photo_lookup = photo_lookup[
    photo_lookup["PhotoNumber"].notna()
].copy()

photo_lookup["PhotoNumber"] = (
    photo_lookup["PhotoNumber"]
    .astype(int)
)


# ============================================================
# PREPARE THE MISSING SAMPLEPOINT RECORDS
# ============================================================

missing_lookup = (
    missing_point1_df
    .copy()
)

missing_lookup["DatabaseNorm"] = (
    missing_lookup["Database"]
    .map(normalize_database)
)

missing_lookup["CameraNorm"] = (
    missing_lookup["CameraAssignment"]
    .map(normalize_camera)
)

missing_lookup["PhotoNumber"] = (
    missing_lookup["Photo"]
    .map(extract_photo_number)
)


# ============================================================
# FIRST MATCH BY PHOTO NUMBER
#
# Then evaluate whether database AND camera agree.
# ============================================================

matches = missing_lookup.merge(
    photo_lookup,
    how="left",
    on="PhotoNumber",
    suffixes=(
        "_Missing",
        "_Survey"
    )
)


# ============================================================
# EVIDENCE CHECKS
# ============================================================

matches["DatabaseMatch"] = (
    matches["DatabaseNorm_Missing"]
    == matches["DatabaseNorm_Survey"]
)

matches["CameraMatch"] = (
    matches["CameraNorm_Missing"]
    == matches["CameraNorm_Survey"]
)

matches["ValidMatch"] = (
    matches["DatabaseMatch"]
    & matches["CameraMatch"]
)


# ============================================================
# KEEP ONLY VALID DATABASE + CAMERA MATCHES
# ============================================================

valid_matches = matches[
    matches["ValidMatch"]
].copy()


# ============================================================
# BUILD RESULT TABLE
# ============================================================

missing_plot_lookup = (
    valid_matches[
        [
            "Database",
            "CameraAssignment",
            "Photo",
            "PhotoNumber",
            "ResolvedPlotID",
            "PhotoPosition",
            "Samplepoint Database Name",
            "Camera Number",
            "ObjectID",
        ]
    ]
    .rename(
        columns={
            "ResolvedPlotID": "PlotID",
            "Samplepoint Database Name": "SurveyDatabase",
            "Camera Number": "SurveyCamera",
        }
    )
    .sort_values(
        [
            "Database",
            "PlotID",
            "PhotoNumber",
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)


print("=" * 90)
print("MISSING PHOTO → PLOT LOOKUP")
print("=" * 90)

print(
    f"\nMissing photos: "
    f"{len(missing_point1_df):,}"
)

print(
    f"Photos with valid plot match: "
    f"{missing_plot_lookup['Photo'].nunique():,}"
)

display(missing_plot_lookup)

MISSING PHOTO → PLOT LOOKUP

Missing photos: 443
Photos with valid plot match: 352


,Database,CameraAssignment,Photo,PhotoNumber,PlotID,PhotoPosition,SurveyDatabase,SurveyCamera,ObjectID
0,CAM5_DATABASE_1,Cam5,DSCN8468_c.jpg,8468,high_32_0,Center Photoplot,Cam5_Database_1,Camera 5,296.0
1,CAM5_DATABASE_1,Cam5,DSCN8469_c.jpg,8469,high_32_0,North Photoplot,Cam5_Database_1,Camera 5,296.0
2,CAM5_DATABASE_1,Cam5,DSCN8470_c.jpg,8470,high_32_0,East Photoplot,Cam5_Database_1,Camera 5,296.0
3,CAM5_DATABASE_1,Cam5,DSCN8471_c.jpg,8471,high_32_0,South Photoplot,Cam5_Database_1,Camera 5,296.0
4,CAM5_DATABASE_1,Cam5,DSCN8472_c.jpg,8472,high_32_0,West Photoplot,Cam5_Database_1,Camera 5,296.0
...,...,...,...,...,...,...,...,...,...
347,JANELLE_DATABASE_6,Cam5,DSCN8448_c.jpg,8448,mid_14_32,Center Photoplot,janelle_database_6,Camera 5,207.0
348,JANELLE_DATABASE_6,Cam5,DSCN8449_c.jpg,8449,mid_14_32,North Photoplot,janelle_database_6,Camera 5,207.0
349,JANELLE_DATABASE_6,Cam5,DSCN8450_c.jpg,8450,mid_14_32,East Photoplot,janelle_database_6,Camera 5,207.0
350,JANELLE_DATABASE_6,Cam5,DSCN8451_c.jpg,8451,mid_14_32,South Photoplot,janelle_database_6,Camera 5,207.0


In [30]:
from pathlib import Path
import pandas as pd

# ============================================================
# UNIQUE PLOTS WITH MISSING SAMPLEPOINT PHOTOS
# ============================================================

missing_plot_ids = (
    missing_plot_lookup["PlotID"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

missing_plot_ids = sorted(missing_plot_ids)

print(f"Unique affected plots: {len(missing_plot_ids):,}")

for plot_id in missing_plot_ids:
    print(plot_id)


# ============================================================
# SUBSET MASTER SURVEY
# ============================================================

MASTER_SURVEY = Path(
    r"D:\My Drive\BOP_OCTC_2025\2026 Master Survey.xlsx"
)

OUTPUT_XLSX = MASTER_SURVEY.with_name(
    "2026 Master Survey_MissingSamplePointPlots.xlsx"
)

survey = pd.read_excel(
    MASTER_SURVEY,
    sheet_name="Sheet1"
)


# Normalize both possible plot-ID fields
survey["_PlotID"] = (
    survey["Plot ID"]
    .fillna("")
    .astype(str)
    .str.strip()
)

survey["_PlotIDIncidental"] = (
    survey["Plot ID Incidental"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# Keep a survey row if EITHER plot-ID field corresponds
# to one of our affected plots.
subset = survey[
    survey["_PlotID"].isin(missing_plot_ids)
    |
    survey["_PlotIDIncidental"].isin(missing_plot_ids)
].copy()


# Remove temporary matching columns
subset = subset.drop(
    columns=[
        "_PlotID",
        "_PlotIDIncidental"
    ]
)


# ============================================================
# QA
# ============================================================

print("\n" + "=" * 72)
print("MISSING-PLOT SURVEY SUBSET")
print("=" * 72)

print(f"Affected PlotIDs requested: {len(missing_plot_ids):,}")
print(f"Survey rows retained:       {len(subset):,}")

# Determine which requested PlotIDs were actually recovered
recovered_ids = set(
    subset["Plot ID"]
    .dropna()
    .astype(str)
    .str.strip()
) | set(
    subset["Plot ID Incidental"]
    .dropna()
    .astype(str)
    .str.strip()
)

not_found = sorted(
    set(missing_plot_ids) - recovered_ids
)

print(f"PlotIDs found in survey:     {len(set(missing_plot_ids) & recovered_ids):,}")
print(f"PlotIDs not found:           {len(not_found):,}")

if not_found:
    print("\nPlotIDs not found:")
    for plot_id in not_found:
        print(" ", plot_id)


# ============================================================
# SAVE
# ============================================================

subset.to_excel(
    OUTPUT_XLSX,
    index=False,
    sheet_name="Missing SamplePoint Plots"
)

print("\nSaved:")
print(OUTPUT_XLSX)

Unique affected plots: 72
20260708_high_34_BRTE
Low_60_ARTR
Mid_19_shrub
Mid_4_hill2
high_32_0
high_32_11
high_32_12
high_32_21
high_32_22
high_34_12
high_35_0
high_35_11
high_35_12
low_60_0
low_60_11
low_60_21
low_77_11
low_77_12
low_77_21
low_77_22
low_77_32
mid_10_11
mid_10_12
mid_10_21
mid_10_22
mid_10_31
mid_14_0
mid_14_11
mid_14_12
mid_14_21
mid_14_22
mid_14_31
mid_14_32
mid_16_0
mid_16_21
mid_16_31
mid_18_11
mid_18_12
mid_18_21
mid_18_31
mid_18_32
mid_19_12
mid_19_13
mid_19_22
mid_19_31
mid_22_13
mid_22_23
mid_22_33
mid_26_0
mid_26_11
mid_26_12
mid_26_21
mid_26_22
mid_26_23
mid_26_31
mid_26_32
mid_26_33
mid_2_0
mid_2_12
mid_4_12
mid_4_13
mid_4_22
mid_4_32
mid_5_12
mid_5_13
mid_5_22
mid_5_23
mid_5_32
mid_8_0
mid_8_12
mid_8_21
mid_8_31

MISSING-PLOT SURVEY SUBSET
Affected PlotIDs requested: 72
Survey rows retained:       72
PlotIDs found in survey:     72
PlotIDs not found:           0

Saved:
D:\My Drive\BOP_OCTC_2025\2026 Master Survey_MissingSamplePointPlots.xlsx


In [31]:
plot_completeness_check = (
    missing_plot_lookup
    .groupby(
        ["PlotID", "Database", "CameraAssignment"],
        as_index=False
    )
    .agg(
        MissingPhotos=("PhotoNumber", "nunique"),
        PhotoNumbers=(
            "PhotoNumber",
            lambda x: ", ".join(
                str(int(v)) for v in sorted(set(x))
            )
        )
    )
    .sort_values(
        ["MissingPhotos", "PlotID"],
        ascending=[True, True]
    )
)

display(plot_completeness_check)

print("\nMissing-photo counts per plot:")
print(
    plot_completeness_check["MissingPhotos"]
    .value_counts()
    .sort_index()
)

,PlotID,Database,CameraAssignment,MissingPhotos,PhotoNumbers
34,mid_16_21,JANELLE_DATABASE_4,AW120,1,9134
12,high_35_12,JANELLE_DATABASE_3,AW120,2,"9039, 9040"
45,mid_22_13,GARRETT_DATABASE_8,AW120,4,"9436, 9437, 9438, 9439"
0,20260708_high_34_BRTE,CAM5_DATABASE_7,Cam5,5,"8770, 8771, 8772, 8773, 8774"
1,Low_60_ARTR,JANELLE_DATABASE_6,Cam5,5,"8368, 8369, 8370, 8371, 8372"
...,...,...,...,...,...
67,mid_5_32,CAM5_DATABASE_6,Cam5,5,"8723, 8724, 8725, 8726, 8727"
68,mid_8_0,CAM5_DATABASE_4,Cam5,5,"8628, 8629, 8630, 8631, 8632"
69,mid_8_12,CAM5_DATABASE_4,Cam5,5,"8633, 8634, 8635, 8636, 8637"
70,mid_8_21,CAM5_DATABASE_4,Cam5,5,"8618, 8619, 8620, 8621, 8622"



Missing-photo counts per plot:
MissingPhotos
1     1
2     1
4     1
5    69
Name: count, dtype: int64


In [32]:
from pathlib import Path
import re
import pandas as pd
import win32com.client as win32


# ============================================================
# VALIDATION XLS DIRECTORY
# ============================================================

VALIDATION_DIR = Path(
    r"D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\validation xls"
)

xls_files_validation = sorted(
    [
        p for p in VALIDATION_DIR.iterdir()
        if p.is_file() and p.suffix.lower() == ".xls"
    ]
)

print(f"Validation XLS files found: {len(xls_files_validation)}")

for p in xls_files_validation:
    print(" ", p.name)


# ============================================================
# HELPERS
# ============================================================

def normalize_header(value):
    if value is None:
        return ""
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower()
    )


def as_2d(value):
    if value is None:
        return []

    if not isinstance(value, tuple):
        return [[value]]

    if value and not isinstance(value[0], tuple):
        return [list(value)]

    return [list(r) for r in value]


def extract_photo_number(value):
    """
    Examples:
        DSCN8418_c.jpg -> 8418
        8418           -> 8418
        8418.0         -> 8418
    """
    if value is None:
        return None

    if isinstance(value, (int, float)) and not isinstance(value, bool):
        try:
            x = float(value)

            if x.is_integer():
                return int(x)

        except (TypeError, ValueError):
            pass

    match = re.search(
        r"(?<!\d)(\d{4,6})(?!\d)",
        str(value)
    )

    return int(match.group(1)) if match else None


def read_validation_xls(excel, path):
    """
    Read first worksheet and return:
        header
        rows
        image_index
        point1_index
    """
    wb = None

    try:
        wb = excel.Workbooks.Open(
            Filename=str(path),
            UpdateLinks=0,
            ReadOnly=True,
            AddToMru=False,
        )

        ws = wb.Worksheets(1)

        values = as_2d(
            ws.UsedRange.Value
        )

        if not values:
            raise ValueError(
                f"No data found in {path.name}"
            )

        # Search first 25 rows for header
        header_row_index = None

        for r_i, row in enumerate(values[:25]):

            norm = [
                normalize_header(v)
                for v in row
            ]

            if (
                "image" in norm
                and "point1" in norm
            ):
                header_row_index = r_i
                break

        if header_row_index is None:
            raise ValueError(
                f"Could not locate Image / Point1 header in {path.name}"
            )

        header = list(
            values[header_row_index]
        )

        header_norm = [
            normalize_header(v)
            for v in header
        ]

        image_index = header_norm.index("image")
        point1_index = header_norm.index("point1")

        rows = []

        for raw_row in values[
            header_row_index + 1:
        ]:

            row = list(raw_row)

            # Pad if needed
            if len(row) < len(header):
                row.extend(
                    [None] * (
                        len(header) - len(row)
                    )
                )

            # Ignore completely blank rows
            if all(
                v is None
                or str(v).strip() == ""
                for v in row
            ):
                continue

            rows.append(row)

        return (
            header,
            rows,
            image_index,
            point1_index
        )

    finally:

        if wb is not None:
            wb.Close(
                SaveChanges=False
            )


# ============================================================
# READ ALL VALIDATION FILES
# ============================================================

records = []

excel = win32.DispatchEx(
    "Excel.Application"
)

excel.Visible = False
excel.DisplayAlerts = False
excel.ScreenUpdating = False

try:

    for path in xls_files_validation:

        (
            header,
            rows,
            image_index,
            point1_index
        ) = read_validation_xls(
            excel,
            path
        )

        n_images = 0

        for row_num, row in enumerate(
            rows,
            start=1
        ):

            image = row[image_index]

            if (
                image is None
                or str(image).strip() == ""
            ):
                continue

            photo_number = extract_photo_number(
                image
            )

            point1 = row[point1_index]

            records.append({
                "ValidationFile": path.stem,
                "Row": row_num,
                "Image": image,
                "PhotoNumber": photo_number,
                "Point1Present": (
                    point1 is not None
                    and str(point1).strip() != ""
                ),
            })

            n_images += 1

        print(
            f"{path.name:25s} "
            f"images={n_images:4d}"
        )

finally:

    excel.ScreenUpdating = True
    excel.DisplayAlerts = True
    excel.Quit()


validation_records = pd.DataFrame(
    records
)


# ============================================================
# COUNT REPEAT OBSERVATIONS
# ============================================================

photo_counts = (
    validation_records
    .groupby(
        "PhotoNumber",
        as_index=False
    )
    .agg(
        N_Observations=(
            "PhotoNumber",
            "size"
        ),
        ValidationFiles=(
            "ValidationFile",
            lambda x: ", ".join(
                sorted(set(x))
            )
        ),
        N_Point1_Present=(
            "Point1Present",
            "sum"
        ),
    )
    .sort_values(
        [
            "N_Observations",
            "PhotoNumber"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("VALIDATION REPEAT-OBSERVATION AUDIT")
print("=" * 80)

print(
    f"\nTotal validation rows with images: "
    f"{len(validation_records):,}"
)

print(
    f"Unique photo numbers: "
    f"{photo_counts['PhotoNumber'].nunique():,}"
)

print("\nObservation-count distribution:")

display(
    photo_counts[
        "N_Observations"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "ObservationsPerPhoto"
    )
    .reset_index(
        name="N_Photos"
    )
)


# ============================================================
# FLAG ANYTHING NOT OBSERVED EXACTLY TWICE
# ============================================================

problems = photo_counts[
    photo_counts[
        "N_Observations"
    ] != 2
].copy()

print("\n" + "=" * 80)
print("PHOTOS NOT OBSERVED EXACTLY TWICE")
print("=" * 80)

print(
    f"\nProblem photo IDs: "
    f"{len(problems):,}"
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(problems)


# ============================================================
# ALSO CHECK WHETHER BOTH REPEATS ACTUALLY HAVE Point1
# ============================================================

incomplete_validation = photo_counts[
    photo_counts[
        "N_Point1_Present"
    ] != photo_counts[
        "N_Observations"
    ]
].copy()

print("\n" + "=" * 80)
print("VALIDATION RECORDS WITH MISSING Point1")
print("=" * 80)

print(
    f"\nPhoto IDs with at least one "
    f"incomplete validation record: "
    f"{len(incomplete_validation):,}"
)

display(incomplete_validation)

Validation XLS files found: 6
  Chloe_GVplots.XLS
  Chloe_JPplots.XLS
  Garrett_CWplots.XLS
  Garrett_JPplots.XLS
  Janelle_CWplots.XLS
  Janelle_GVplots.XLS
Chloe_GVplots.XLS         images=  75
Chloe_JPplots.XLS         images=  75
Garrett_CWplots.XLS       images=  75
Garrett_JPplots.XLS       images=  75
Janelle_CWplots.XLS       images=  75
Janelle_GVplots.XLS       images=  75

VALIDATION REPEAT-OBSERVATION AUDIT

Total validation rows with images: 450
Unique photo numbers: 225

Observation-count distribution:


,ObservationsPerPhoto,N_Photos
0,2,225



PHOTOS NOT OBSERVED EXACTLY TWICE

Problem photo IDs: 0


,PhotoNumber,N_Observations,ValidationFiles,N_Point1_Present



VALIDATION RECORDS WITH MISSING Point1

Photo IDs with at least one incomplete validation record: 70


,PhotoNumber,N_Observations,ValidationFiles,N_Point1_Present
0,105,2,"Chloe_GVplots, Janelle_GVplots",1
1,106,2,"Chloe_GVplots, Janelle_GVplots",1
2,107,2,"Chloe_GVplots, Janelle_GVplots",1
3,108,2,"Chloe_GVplots, Janelle_GVplots",1
4,109,2,"Chloe_GVplots, Janelle_GVplots",1
...,...,...,...,...
160,9310,2,"Chloe_GVplots, Janelle_GVplots",1
161,9311,2,"Chloe_GVplots, Janelle_GVplots",1
162,9312,2,"Chloe_GVplots, Janelle_GVplots",1
163,9313,2,"Chloe_GVplots, Janelle_GVplots",1


In [33]:
# ============================================================
# WHICH VALIDATION XLS FILES CONTAIN INCOMPLETE ROWS?
# ============================================================

validation_incompletes = (
    validation_records[
        validation_records["Point1Present"] == False
    ]
    .copy()
)

print(
    f"Total incomplete validation rows: "
    f"{len(validation_incompletes):,}"
)

# ------------------------------------------------------------
# Summary by XLS
# ------------------------------------------------------------

incomplete_by_xls = (
    validation_incompletes
    .groupby(
        "ValidationFile",
        as_index=False
    )
    .agg(
        IncompleteRows=(
            "PhotoNumber",
            "size"
        ),
        UniquePhotos=(
            "PhotoNumber",
            "nunique"
        ),
        PhotoNumbers=(
            "PhotoNumber",
            lambda x: ", ".join(
                str(int(v))
                for v in sorted(
                    set(x.dropna())
                )
            )
        ),
    )
    .sort_values(
        "IncompleteRows",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nValidation XLS files containing incompletes:")
display(incomplete_by_xls)


# ------------------------------------------------------------
# Full row-level detail
# ------------------------------------------------------------

print("\nIndividual incomplete records:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(
        validation_incompletes[
            [
                "ValidationFile",
                "Row",
                "Image",
                "PhotoNumber",
            ]
        ]
        .sort_values(
            [
                "ValidationFile",
                "PhotoNumber"
            ]
        )
        .reset_index(drop=True)
    )

Total incomplete validation rows: 70

Validation XLS files containing incompletes:


,ValidationFile,IncompleteRows,UniquePhotos,PhotoNumbers
0,Janelle_CWplots,35,35,"8202, 8203, 8204, 8205, 8206, 8207, 8208, 8209..."
1,Janelle_GVplots,35,35,"105, 106, 107, 108, 109, 8916, 8917, 8918, 891..."



Individual incomplete records:


,ValidationFile,Row,Image,PhotoNumber
0,Janelle_CWplots,6,DSCN8202_c.jpg,8202
1,Janelle_CWplots,7,DSCN8203_c.jpg,8203
2,Janelle_CWplots,8,DSCN8204_c.jpg,8204
3,Janelle_CWplots,9,DSCN8205_c.jpg,8205
4,Janelle_CWplots,10,DSCN8206_c.jpg,8206
5,Janelle_CWplots,11,DSCN8207_c.jpg,8207
6,Janelle_CWplots,12,DSCN8208_c.jpg,8208
7,Janelle_CWplots,13,DSCN8209_c.jpg,8209
8,Janelle_CWplots,14,DSCN8210_c.jpg,8210
9,Janelle_CWplots,15,DSCN8211_c.jpg,8211
